## Import Libraries

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

C:\Users\kavya\AppData\Local\Temp\ipykernel_7992\4087828706.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Configure

In [2]:
load_dotenv()

DATA_DIR = Path("data")
VECTORSTORE_DIR = Path("vectorstores")

FAISS_DIR = VECTORSTORE_DIR / "faiss_index"
CHROMA_DIR = VECTORSTORE_DIR / "chroma_db"

DATA_DIR.mkdir(exist_ok=True)
VECTORSTORE_DIR.mkdir(exist_ok=True)

print("Project configured successfully.")

Project configured successfully.


## Configure the models

In [32]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

print("Models configured successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2210.24it/s]


Models configured successfully.


## Load the Documents

In [4]:
documents = []

for pdf_file in DATA_DIR.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_file))
    documents.extend(loader.load())

print("Total pages loaded:", len(documents))

Total pages loaded: 87


## Split Documents

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))

Total chunks created: 277


#### Simple Check

In [6]:
if not documents:
    print("No PDFs found. Add PDF files to the data folder.")
elif not chunks:
    print("No text chunks were created. Check the PDF contents.")
else:
    print("PDF loading and chunking completed successfully.")

PDF loading and chunking completed successfully.


## Langchain

In [7]:

# Version 1: Simple LangChain without RAG

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

simple_prompt = ChatPromptTemplate.from_template("""
You are a helpful academic assistant.

Answer the following question clearly and concisely.

Question: {question}

Answer:
""")

simple_chain = simple_prompt | llm | StrOutputParser()

print("Simple LangChain chain created successfully.")

Simple LangChain chain created successfully.


In [8]:

question = "What is RAG in Generative AI?"

answer = simple_chain.invoke({
    "question": question
})

print("Question:", question)
print("\nAnswer:", answer)

Question: What is RAG in Generative AI?

Answer: **RAG** stands for **Retrieval-Augmented Generation**. It is an architectural framework in generative AI that improves the accuracy and reliability of Large Language Models (LLMs) by integrating an external information retrieval mechanism into the generation process.

### How It Works:
1. **Retrieval:** When a user submits a query, the system searches an external knowledge source (such as documents, proprietary databases, or the web)—often using vector embeddings—to find contextually relevant information.
2. **Augmentation:** The retrieved information is appended to the user’s original prompt as reference context.
3. **Generation:** The LLM reads the enriched prompt (query + retrieved data) and synthesizes a tailored, accurate response.

### Key Benefits:
* **Mitigates Hallucinations:** Grounds model outputs in verifiable, factual data.
* **Access to Current/Private Data:** Allows the model to reference domain-specific or up-to-date info

## FAISS

In [9]:
if FAISS_DIR.exists() and (FAISS_DIR / "index.faiss").exists():
    vector_store = FAISS.load_local(
        str(FAISS_DIR),
        embedding_model,
        allow_dangerous_deserialization=True
    )
    print("Existing FAISS index loaded.")
else:
    vector_store = FAISS.from_documents(
        chunks,
        embedding_model
    )

    vector_store.save_local(str(FAISS_DIR))
    print("New FAISS index created and saved.")

Existing FAISS index loaded.


## Retriever

In [10]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

In [11]:
prompt = ChatPromptTemplate.from_template("""
You are a JNTUH Academic Support Assistant.

Answer the question using only the provided context.
If the answer is not available in the context, say that you could not
find the information in the provided JNTUH documents.

Context:
{context}

Question:
{question}

Answer:
""")

In [33]:
rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

In [13]:
question = "What are the academic regulations for B.Tech students?"

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

answer = rag_chain.invoke({
    "context": context,
    "question": question
})

print(answer)

Based on the provided context, the academic regulations for B.Tech students include:

* **Medium of Instruction:** English only for the entire undergraduate programme in Engineering & Technology.
* **Programme Duration:** 
  * A student must complete the B.Tech programme in a minimum of four academic years and a maximum of eight academic years from the date of commencement of the first year first semester; otherwise, the student forfeits their seat.
  * For Lateral Entry Scheme (LES) students, requirements for the degree must be fulfilled within six academic years, or the seat is forfeited.
* **Credit Requirements and Award of Degree:** 
  * A student must register for all specified courses and secure a minimum of 160 credits out of 164 credits within 8 academic years for the award of the B.Tech degree.
  * The best 160 credits are considered for CGPA computation.
  * Students can avail of an exemption of courses totaling up to 4 credits (excluding Professional core courses, Laboratory

In [14]:
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\nSource {i}")
    print("File:", doc.metadata.get("source", "Unknown"))
    print("Page:", doc.metadata.get("page", "Unknown"))
    print("Content:", doc.page_content[:300])


Source 1
File: data\R25B.Tech.AcademicRegulationsV2.pdf
Page: 16
Content: examinations or not. 
3 Third year first semester to 
Third year second semester 
Regular course of study of third year first semester 
and fulfilment of attendance requirement. 
4 Third year second semester to 
Fourth year first semester 
Regular course of study of third year second 
semester and f

Source 2
File: data\R25B.Tech.AcademicRegulationsV2.pdf
Page: 0
Content: Government (EAPCET) or the University or on the basis of any other order of merit approved 
by the University, subject to reservations as prescribed by the government from time to time. 
2.2 The medium of instruction for the entire undergraduate programme in Engineering & 
Technology will be English

Source 3
File: data\R25B.Tech.AcademicRegulationsV2.pdf
Page: 16
Content: 4. The students, who fail to fulfil the requirement for the award of the degree in six academic years 
from the year of admission, shall forfeit their seat in B.Tech. 
5. The

## CHROMADB

In [15]:
if CHROMA_DIR.exists() and any(CHROMA_DIR.iterdir()):
    chroma_store = Chroma(
        collection_name="jntuh_documents",
        embedding_function=embedding_model,
        persist_directory=str(CHROMA_DIR)
    )
    print("Existing Chroma database loaded.")
else:
    chroma_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        collection_name="jntuh_documents",
        persist_directory=str(CHROMA_DIR)
    )
    print("New Chroma database created and saved.")

Existing Chroma database loaded.


In [16]:
chroma_retriever = chroma_store.as_retriever(
    search_kwargs={"k": 4}
)

In [17]:
question = "What are the academic regulations for B.Tech students?"

chroma_docs = chroma_retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in chroma_docs
)

answer = rag_chain.invoke({
    "context": context,
    "question": question
})

print(answer)

Based on the provided JNTUH documents, the academic regulations for B.Tech students include:

* **Medium of Instruction:** The medium of instruction for the entire undergraduate programme is English only.
* **Programme Duration:** 
  * Regular B.Tech students must complete the programme in a minimum of 4 academic years and a maximum of 8 academic years from the date of commencement of the first year first semester, failing which they forfeit their seat.
  * Lateral Entry Scheme (LES) students must fulfill the requirements within 6 academic years from the year of admission or forfeit their seat.
* **Credit Requirements and Award of Degree:**
  * Students must register for the course structure and secure a minimum of 160 credits out of 164 credits within 8 academic years for the award of the B.Tech degree.
  * The best 160 credits are considered for CGPA computation. Students may avail an exemption of up to 4 credits (excluding Professional core courses, Laboratory courses, Seminars, Pro

In [18]:
for i, doc in enumerate(chroma_docs, start=1):
    print(f"\nSource {i}")
    print("File:", doc.metadata.get("source", "Unknown"))
    print("Page:", doc.metadata.get("page", "Unknown"))
    print("Content:", doc.page_content[:300])


Source 1
File: data\R25B.Tech.AcademicRegulationsV2.pdf
Page: 16
Content: examinations or not. 
3 Third year first semester to 
Third year second semester 
Regular course of study of third year first semester 
and fulfilment of attendance requirement. 
4 Third year second semester to 
Fourth year first semester 
Regular course of study of third year second 
semester and f

Source 2
File: data\R25B.Tech.AcademicRegulationsV2.pdf
Page: 0
Content: Government (EAPCET) or the University or on the basis of any other order of merit approved 
by the University, subject to reservations as prescribed by the government from time to time. 
2.2 The medium of instruction for the entire undergraduate programme in Engineering & 
Technology will be English

Source 3
File: data\R25B.Tech.AcademicRegulationsV2.pdf
Page: 16
Content: 4. The students, who fail to fulfil the requirement for the award of the degree in six academic years 
from the year of admission, shall forfeit their seat in B.Tech. 
5. The

## RE-Ranking

In [19]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranking model loaded.")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2035.49it/s]


Reranking model loaded.


#### REtrieve and rerank chunks

In [20]:
question = "What are the academic regulations for B.Tech students?"

candidate_docs = vector_store.similarity_search(
    question,
    k=15
)

pairs = [
    [question, doc.page_content]
    for doc in candidate_docs
]

scores = reranker.predict(pairs)

ranked_docs = sorted(
    zip(candidate_docs, scores),
    key=lambda item: item[1],
    reverse=True
)

advanced_docs = [
    doc for doc, score in ranked_docs[:4]
]

print("Candidate chunks:", len(candidate_docs))
print("Chunks after reranking:", len(advanced_docs))

Candidate chunks: 15
Chunks after reranking: 4


In [21]:
advanced_context = "\n\n".join(
    doc.page_content for doc in advanced_docs
)

advanced_answer = rag_chain.invoke({
    "context": advanced_context,
    "question": question
})

print(advanced_answer)

Based on the provided JNTUH documents, the academic regulations for B.Tech students include the following:

* **Regulations Framework:** Known as the **R-25 Regulations**, offered under the Choice Based Credit System (CBCS) effective from the academic year 2025-26 for regular students and from 2026-27 for Lateral Entry Scheme (LES) students.
* **Medium of Instruction:** The medium of instruction for the entire undergraduate programme is English only.
* **Eligibility for Admission:** Admissions are made based on the merit rank obtained in the entrance test conducted by the Telangana Government (EAPCET), by the University, or other approved orders of merit, subject to government reservations.
* **Programme Duration:**
  * **Regular B.Tech:** A minimum period of 4 academic years (eight semesters) and a maximum period of 8 academic years. Failing to complete within 8 years results in forfeiture of the seat.
  * **Lateral Entry Scheme (LES):** Not less than 3 academic years and not more tha

In [22]:
for i, (doc, score) in enumerate(ranked_docs[:4], start=1):
    print(f"\nRank {i} | Score: {score:.4f}")
    print("File:", doc.metadata.get("source", "Unknown"))
    print("Page:", doc.metadata.get("page", "Unknown"))
    print("Content:", doc.page_content[:300])


Rank 1 | Score: 7.5957
File: data\R25B.Tech.AcademicRegulationsV2.pdf
Page: 0
Content: 1 
 
 
JAWAHARLAL NEHRU TECHNOLOGICAL UNIVERSITY HYDERABAD 
(Established by State Act No. 30 of 2008) 
Kukatpally, Hyderabad, Telangana (India) 
 
ACADEMIC REGULATIONS FOR B. TECH. REGULAR STUDENTS  
WITH EFFECT FROM THE ACADEMIC YEAR 2025-26 (R-25) 
 
1.0 Under-Graduate Degree Programme in Engineer

Rank 2 | Score: 7.5035
File: data\R25B.Tech.AcademicRegulationsV2.pdf
Page: 16
Content: 17 
 
 
JAWAHARLAL NEHRU TECHNOLOGICAL UNIVERSITY HYDERABAD 
(Established by State Act No. 30 of 2008) 
Kukatpally, Hyderabad, Telangana (India) 
ACADEMIC REGULATIONS FOR B.TECH. (LATERAL ENTRY SCHEME) FROM THE AY 2026-27 
 
Eligibility for the award of B.Tech. Degree (LES) 
1. The LES students afte

Rank 3 | Score: 6.1442
File: data\R25B.Tech.AcademicRegulationsV2.pdf
Page: 0
Content: Government (EAPCET) or the University or on the basis of any other order of merit approved 
by the University, subject to reservation

In [23]:
import time

question = "What are the academic regulations for B.Tech students?"

# Basic RAG with FAISS
start = time.time()
faiss_docs = retriever.invoke(question)
faiss_context = "\n\n".join(doc.page_content for doc in faiss_docs)
faiss_answer = rag_chain.invoke({
    "context": faiss_context,
    "question": question
})
faiss_time = time.time() - start

# Basic RAG with Chroma
start = time.time()
chroma_docs = chroma_retriever.invoke(question)
chroma_context = "\n\n".join(doc.page_content for doc in chroma_docs)
chroma_answer = rag_chain.invoke({
    "context": chroma_context,
    "question": question
})
chroma_time = time.time() - start

# Advanced RAG with FAISS and reranking
start = time.time()
candidate_docs = vector_store.similarity_search(question, k=15)
pairs = [[question, doc.page_content] for doc in candidate_docs]
scores = reranker.predict(pairs)

ranked_docs = sorted(
    zip(candidate_docs, scores),
    key=lambda item: item[1],
    reverse=True
)
advanced_docs = [doc for doc, score in ranked_docs[:4]]
advanced_context = "\n\n".join(doc.page_content for doc in advanced_docs)
advanced_answer = rag_chain.invoke({
    "context": advanced_context,
    "question": question
})
advanced_time = time.time() - start

print("FAISS RAG:", round(faiss_time, 2), "seconds")
print("Chroma RAG:", round(chroma_time, 2), "seconds")
print("Advanced RAG:", round(advanced_time, 2), "seconds")

FAISS RAG: 10.95 seconds
Chroma RAG: 282.08 seconds
Advanced RAG: 108.86 seconds


In [34]:
def faiss_rag(question):
    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    answer = rag_chain.invoke({
        "context": context,
        "question": question
    })

    return answer, docs

In [35]:
def chroma_rag(question):
    docs = chroma_retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    answer = rag_chain.invoke({
        "context": context,
        "question": question
    })

    return answer, docs

In [36]:
def advanced_rag(question):
    candidate_docs = vector_store.similarity_search(
        question,
        k=15
    )

    pairs = [
        [question, doc.page_content]
        for doc in candidate_docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(candidate_docs, scores),
        key=lambda item: item[1],
        reverse=True
    )

    docs = [
        doc for doc, score in ranked_docs[:4]
    ]

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    answer = rag_chain.invoke({
        "context": context,
        "question": question
    })

    return answer, docs

In [27]:
question = "What are the academic regulations for B.Tech students?"

faiss_answer, faiss_sources = faiss_rag(question)
chroma_answer, chroma_sources = chroma_rag(question)
advanced_answer, advanced_sources = advanced_rag(question)

print("BASIC RAG - FAISS")
print(faiss_answer)

print("\nBASIC RAG - CHROMA")
print(chroma_answer)

print("\nADVANCED RAG")
print(advanced_answer)

BASIC RAG - FAISS
Based on the provided context, the academic regulations for B.Tech students include:

* **Medium of Instruction:** The medium of instruction for the entire undergraduate programme is English only.
* **Programme Duration:** 
  * A regular student must complete the B.Tech. programme in a minimum of 4 academic years and a maximum of 8 academic years from the date of commencement of the first year first semester, failing which they forfeit their seat.
  * For Lateral Entry Scheme (LES) students, the degree requirements must be fulfilled within 6 academic years from the year of admission, failing which they forfeit their seat.
* **Credits and Award of Degree:**
  * Students must register for all specified courses and secure a minimum of 160 credits out of 164 credits for the award of the degree.
  * The best 160 credits are considered for CGPA computation. 
  * Students can avail of course exemptions totaling up to 4 credits (optional drop out from the 164 registered credi

## Test Questions

In [28]:
test_questions = [
    "What are the academic regulations for B.Tech students?",
    "What are the attendance requirements for students?",
    "What are the rules regarding examinations?",
    "What is the procedure for evaluating students?"
]

In [37]:
import time
import pandas as pd

results = []

for question in test_questions:

    # Basic RAG - FAISS
    start = time.time()
    faiss_answer, faiss_sources = faiss_rag(question)
    faiss_time = time.time() - start

    # Basic RAG - Chroma
    start = time.time()
    chroma_answer, chroma_sources = chroma_rag(question)
    chroma_time = time.time() - start

    # Advanced RAG
    start = time.time()
    advanced_answer, advanced_sources = advanced_rag(question)
    advanced_time = time.time() - start

    results.append({
        "Question": question,
        "FAISS Time (s)": round(faiss_time, 2),
        "Chroma Time (s)": round(chroma_time, 2),
        "Advanced RAG Time (s)": round(advanced_time, 2),
        "FAISS Sources": len(faiss_sources),
        "Chroma Sources": len(chroma_sources),
        "Advanced Sources": len(advanced_sources)
    })

results_df = pd.DataFrame(results)

results_df

c:\Users\kavya\OneDrive\Desktop\RAG\rag_proj\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\kavya\OneDrive\Desktop\RAG\rag_proj\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\kavya\OneDrive\Desktop\RAG\rag_proj\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\kavya\OneDrive\Desktop\RAG\rag_proj\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling pa

,Question,FAISS Time (s),Chroma Time (s),Advanced RAG Time (s),FAISS Sources,Chroma Sources,Advanced Sources
0,What are the academic regulations for B.Tech s...,12.03,7.38,11.39,4,4,4
1,What are the attendance requirements for stude...,6.64,6.54,26.42,4,4,4
2,What are the rules regarding examinations?,18.43,8.32,7.56,4,4,4
3,What is the procedure for evaluating students?,10.96,12.18,7.78,4,4,4


In [38]:
for question in test_questions:
    print("\n" + "=" * 80)
    print("QUESTION:", question)

    answer, sources = faiss_rag(question)
    print("\nFAISS ANSWER:\n", answer)

    answer, sources = chroma_rag(question)
    print("\nCHROMA ANSWER:\n", answer)

    answer, sources = advanced_rag(question)
    print("\nADVANCED RAG ANSWER:\n", answer)


QUESTION: What are the academic regulations for B.Tech students?


c:\Users\kavya\OneDrive\Desktop\RAG\rag_proj\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



FAISS ANSWER:
 Based on the provided documents, the academic regulations for B.Tech students are as follows:

### **1. Medium of Instruction**
* The medium of instruction for the entire undergraduate program is **English only**.

### **2. Program Duration and Seat Forfeiture**
* **Regular B.Tech:** Must be completed in a minimum period of **4 academic years** and a maximum period of **8 academic years** from the date of commencement of the first year first semester. Failure to complete within 8 years results in forfeiture of the seat.
* **Lateral Entry Scheme (LES):** Must fulfill the requirements for the award of the degree within **6 academic years** from the year of admission, failing which the seat is forfeited.

### **3. Credit Requirements and Award of Degree**
* **Total Credits Required:** Students must secure a minimum of **160 credits out of 164 credits** registered.
* **CGPA Computation:** The best 160 credits are considered for CGPA computation.
* **Credit Exemption:** Stud

c:\Users\kavya\OneDrive\Desktop\RAG\rag_proj\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



CHROMA ANSWER:
 Based on the provided JNTUH documents, the academic regulations for B.Tech students are as follows:

### **1. Medium of Instruction**
* The medium of instruction for the entire undergraduate program is **English only**.

### **2. Program Duration and Seat Forfeiture**
* **Regular Students:** Must complete the B.Tech program in a minimum period of 4 academic years and a maximum period of 8 academic years from the date of commencement of the first year first semester. Failure to do so results in seat forfeiture.
* **Lateral Entry Scheme (LES) Students:** Must fulfill the degree requirements within 6 academic years from the year of admission, failing which they forfeit their seat.

### **3. Credit Requirements and Award of Degree**
* **Total Credits Required:** A student must secure a minimum of **160 credits out of 164 registered credits** to be awarded the B.Tech degree.
* **CGPA Computation:** The best 160 credits are considered for CGPA computation.
* **Credit Exempti

c:\Users\kavya\OneDrive\Desktop\RAG\rag_proj\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 17h27m19.526781749s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

In [ ]:
def ask_question(question, rag_version):

    if rag_version == "FAISS":
        answer, sources = faiss_rag(question)

    elif rag_version == "Chroma":
        answer, sources = chroma_rag(question)

    elif rag_version == "Advanced RAG":
        answer, sources = advanced_rag(question)

    else:
        return "Invalid RAG version selected.", []

    return answer, sources

In [ ]:
question = "What are the attendance requirements for students?"

answer, sources = ask_question(
    question,
    "Advanced RAG"
)

print("ANSWER:")
print(answer)

print("\nSOURCES:")
for doc in sources:
    print("File:", doc.metadata.get("source", "Unknown"))
    print("Page:", doc.metadata.get("page", "Unknown"))